# 01 — Exploración de datos: Sen1Floods11

Objetivo: **entender qué contiene cada archivo antes de modelar**.

| Archivo | Qué es | Rol en el proyecto |
|---|---|---|
| `splits/*.csv` | Lista de chips de cada partición | Separar train / valid / test / Bolivia |
| `Sen1Floods11_Metadata.geojson` | Un registro por evento de inundación | Contexto (fechas, órbitas, umbrales) |
| `S1Hand/*.tif` | Imagen de radar, 2 bandas (VV, VH) en dB | **Entrada** del modelo |
| `LabelHand/*.tif` | Máscara dorada: 1 agua, 0 no agua, -1 sin dato | **Respuesta correcta** |

## 0. Preparación

In [ ]:
!pip install -q rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

DATA = '/content/drive/MyDrive/tarea2_floods/data/sen1floods11'
pd.set_option('display.max_columns', 30)

## 1. Particiones (`splits/*.csv`)

Los CSV **no tienen encabezado**: son 2 columnas sin nombre. Al leerlos les ponemos nombre:
- `s1_file`: imagen de radar del chip
- `label_file`: su máscara

El nombre del archivo es `<evento>_<id del chip>_<tipo>.tif`.

In [ ]:
train = pd.read_csv(f'{DATA}/splits/flood_train_data.csv', header=None, names=['s1_file', 'label_file'])
train.head()

In [ ]:
train.info()

Juntamos las 4 particiones en una sola tabla, y agregamos columnas con el **evento** y el **id** sacados del nombre del archivo.

In [ ]:
splits = []
for s in ['train', 'valid', 'test', 'bolivia']:
    df = pd.read_csv(f'{DATA}/splits/flood_{s}_data.csv', header=None, names=['s1_file', 'label_file'])
    df['split'] = s
    splits.append(df)
splits = pd.concat(splits, ignore_index=True)

splits['evento']  = splits['s1_file'].str.split('_').str[0]
splits['chip_id'] = splits['s1_file'].str.split('_').str[1]
splits.head()

In [ ]:
splits['split'].value_counts()      # esperado: train 252, test 90, valid 89, bolivia 15

**Chips por evento y partición.** Fíjate en que Bolivia solo aparece en `bolivia`, porque es la región que se reserva para medir generalización.

In [ ]:
pd.crosstab(splits['evento'], splits['split'], margins=True)

## 2. Metadata de eventos (`Sen1Floods11_Metadata.geojson`)

Un registro por evento (12). Columnas:

| Columna | Significado |
|---|---|
| `ID`, `location`, `ISO_CC` | Número, nombre y código del país del evento |
| `s1_date` / `s2_date` | Fecha de la imagen de radar (Sentinel-1) y de la óptica (Sentinel-2) |
| `orbit`, `rel_orbit_num` | Dirección del satélite (ASCENDING/DESCENDING) y número de órbita |
| `VH_thresh` | Umbral (dB, banda VH) usado para generar las etiquetas **débiles**: más oscuro que esto = agua |
| `train_chip` | ⚠️ En realidad es el nº de chips **débiles** del evento |
| `val_chip` | ⚠️ En realidad es el nº de chips **dorados** del evento |
| `coincident_size` | No documentado; no lo usamos |
| `geometry` | Polígono con la zona del evento |

Nota: **"Mekong" en los nombres de archivo = Cambodia** en la metadata.

In [ ]:
meta = gpd.read_file(f'{DATA}/Sen1Floods11_Metadata.geojson')
meta.head()

In [ ]:
meta.info()

In [ ]:
meta.describe()

Tabla resumida, ordenada por fecha del radar. `VH_thresh` **cambia según el evento**: no hay un umbral universal para detectar agua.

In [ ]:
meta[['location', 'ISO_CC', 's1_date', 'orbit', 'VH_thresh', 'val_chip', 'train_chip']] \
    .rename(columns={'val_chip': 'chips_dorados', 'train_chip': 'chips_debiles'}) \
    .sort_values('s1_date')

## 3. Un chip de radar por dentro (`S1Hand`)

Un GeoTIFF no tiene columnas sino **bandas**: matrices de 512×512 apiladas.
- **VV** (banda 1): emite vertical, recibe vertical. Muy sensible al agua.
- **VH** (banda 2): emite vertical, recibe horizontal. Sensible a la vegetación; siempre más oscura.
- Unidad: **dB**. Muy negativo = oscuro = agua probable; ~-10 = suelo/vegetación; positivo = ciudad.

In [ ]:
chip = 'Ghana_103272'
src = rasterio.open(f'{DATA}/S1Hand/{chip}_S1Hand.tif')

print('Bandas      :', src.count, src.descriptions)
print('Tamaño      :', src.height, 'x', src.width)
print('Tipo de dato:', src.dtypes)
print('Coordenadas :', src.crs)
print('Resolución  :', src.res, '(grados ≈ 10 m)')
print('Sin dato    :', src.nodata)
print('Límites     :', src.bounds)

Convertimos los píxeles en una **tabla**: cada fila es un píxel y cada columna es una banda (512 × 512 = 262,144 filas).

In [ ]:
img = src.read()                      # forma (2, 512, 512)
pix = pd.DataFrame({'VV': img[0].ravel(), 'VH': img[1].ravel()})
pix.head()

In [ ]:
pix.describe()

In [ ]:
pix.isna().sum()                      # píxeles sin dato (NaN)

## 4. La máscara dorada (`LabelHand`)

1 banda, valores enteros:

| Valor | Significado |
|---|---|
| 1 | Agua |
| 0 | No agua |
| -1 | **Sin dato** (nubes o zona dudosa). ⚠️ El archivo **no** lo declara como "sin dato": hay que excluirlo a mano en la loss y las métricas |

In [ ]:
lab = rasterio.open(f'{DATA}/LabelHand/{chip}_LabelHand.tif').read(1)
pd.Series(lab.ravel()).value_counts().rename_axis('valor').to_frame('pixeles')

In [ ]:
pd.Series(lab.ravel()).value_counts(normalize=True).mul(100).round(2).rename_axis('valor').to_frame('% pixeles')

Juntamos imagen + máscara en la misma tabla y comparamos los dB del agua con los del resto.

In [ ]:
pix['label'] = lab.ravel()
pix.groupby('label')[['VV', 'VH']].describe().T

## 5. Inventario de los 446 chips

Una fila por chip con sus estadísticas. Esta tabla es la base de toda la exploración (toma 1–2 minutos porque lee 892 archivos desde Drive).

| Columna | Significado |
|---|---|
| `pct_agua` | % de píxeles de agua (sobre los píxeles válidos) |
| `pct_sin_dato` | % de píxeles con -1 en la máscara |
| `pct_nan` | % de píxeles NaN en la imagen de radar |
| `vv_agua`, `vv_tierra` | dB promedio de VV en agua y en no agua |
| `vh_agua`, `vh_tierra` | Lo mismo para VH |

In [ ]:
rows = []
for r in splits.itertuples():
    img = rasterio.open(f'{DATA}/S1Hand/{r.s1_file}').read()
    lab = rasterio.open(f'{DATA}/LabelHand/{r.label_file}').read(1)
    agua, tierra = lab == 1, lab == 0
    rows.append({
        'chip_id': r.chip_id, 'evento': r.evento, 'split': r.split,
        'pct_agua': 100 * agua.sum() / max((lab != -1).sum(), 1),
        'pct_sin_dato': 100 * (lab == -1).mean(),
        'pct_nan': 100 * np.isnan(img[0]).mean(),
        'vv_agua': np.nanmean(img[0][agua]) if agua.any() else np.nan,
        'vv_tierra': np.nanmean(img[0][tierra]) if tierra.any() else np.nan,
        'vh_agua': np.nanmean(img[1][agua]) if agua.any() else np.nan,
        'vh_tierra': np.nanmean(img[1][tierra]) if tierra.any() else np.nan,
    })
chips = pd.DataFrame(rows)
chips.head()

In [ ]:
chips.info()

In [ ]:
chips.describe().round(2)

**Resumen por evento:** cuánta agua hay, cuántos datos faltan y si el agua se ve distinta en cada lugar.

In [ ]:
chips.groupby('evento').agg(
    chips=('chip_id', 'count'),
    pct_agua_media=('pct_agua', 'mean'),
    chips_sin_agua=('pct_agua', lambda x: (x == 0).sum()),
    pct_sin_dato_media=('pct_sin_dato', 'mean'),
    vv_agua=('vv_agua', 'mean'),
    vv_tierra=('vv_tierra', 'mean'),
).round(2).sort_values('pct_agua_media', ascending=False)

In [ ]:
chips.groupby('split')[['pct_agua', 'pct_sin_dato', 'pct_nan']].mean().round(2)

In [ ]:
# Guardar el inventario para usarlo en las siguientes secciones y en el notebook de modelado
chips.to_csv(f'{DATA}/inventario_chips.csv', index=False)

## Qué anotar de esta exploración (para el informe)

- ¿Qué % de píxeles es agua? → justifica usar IoU/F1 en vez de accuracy.
- ¿Cuántos chips no tienen agua? → el modelo también debe aprender a decir "aquí no hay agua".
- ¿Cuánto -1 y NaN hay? → confirma que hay que enmascararlos.
- ¿El agua es claramente más oscura que la tierra en VV y VH? ¿Cambia por evento? → motiva un modelo en vez de un umbral fijo.